In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# ----------------------------
# Paths
# ----------------------------
data_path = Path("../data/raw/2021_2023")
output_path = Path("../data/final/nhanes_2021_2023.csv")

files = {
    "DEMO": "DEMO_L.xpt",
    "DIQ": "DIQ_L.xpt",
    "KIQ": "KIQ_U_L.xpt",
    "MCQ": "MCQ_L.xpt"
}

# ----------------------------
# NHANES Loader (Fix byte columns)
# ----------------------------
def load_nhanes_xpt(path):
    df = pd.read_sas(path, format="xport")
    df.columns = [
        col.decode("utf-8") if isinstance(col, bytes) else col
        for col in df.columns
    ]
    return df

# ----------------------------
# Load datasets
# ----------------------------
dfs = {k: load_nhanes_xpt(data_path / v) for k, v in files.items()}

# ----------------------------
# Merge on SEQN
# ----------------------------
merged_df = dfs["DEMO"]

for key in dfs:
    if key != "DEMO":
        merged_df = merged_df.merge(dfs[key], on="SEQN", how="left")

print("Merged shape:", merged_df.shape)

# ----------------------------
# Remove SAS tiny numeric artifacts
# ----------------------------
numeric_cols = merged_df.select_dtypes(include=[np.number]).columns

merged_df[numeric_cols] = merged_df[numeric_cols].mask(
    merged_df[numeric_cols].abs() < 1e-10,
    np.nan
)


# ----------------------------
# Ensure OAB columns exist
# ----------------------------
oab_cols = ["KIQ005", "KIQ044", "KIQ481"]

missing = [c for c in oab_cols if c not in merged_df.columns]
if missing:
    raise ValueError(f"Missing required OAB columns: {missing}")

# ----------------------------
# Restrict to participants asked urinary questions
# ----------------------------
merged_df = merged_df[
    merged_df[oab_cols].notna().any(axis=1)
]

print("After restricting to urinary respondents:", merged_df.shape)

# ----------------------------
# OAB Labelling
# ----------------------------
def valid_symptom(val):
    if pd.isna(val):
        return False
    if val in [7, 9, 77, 99]:
        return False
    return True

# ----------------------------
# Replace NHANES missing codes
# ----------------------------
merged_df.replace([7, 9, 77, 99, 777, 999], np.nan, inplace=True)

# ----------------------------
# Drop sparse predictor columns
# (Never drop label columns)
# ----------------------------
threshold = 0.5 * len(merged_df)

cols_to_drop = [
    col for col in merged_df.columns
    if merged_df[col].notna().sum() < threshold
    and col not in oab_cols
    and col != "OAB"
]

merged_df = merged_df.drop(columns=cols_to_drop)

print("After dropping sparse predictors:", merged_df.shape)

# ----------------------------
# Missing value imputation
# ----------------------------
for col in merged_df.columns:

    if merged_df[col].dtype.kind in "biufc":

        median_val = merged_df[col].median()

        if pd.isna(median_val):
            merged_df[col] = merged_df[col].fillna(0)
        else:
            merged_df[col] = merged_df[col].fillna(median_val)

    else:
        mode_vals = merged_df[col].mode()

        if len(mode_vals) == 0:
            merged_df[col] = merged_df[col].fillna("Unknown")
        else:
            merged_df[col] = merged_df[col].fillna(mode_vals[0])

# ----------------------------
# Final Check
# ----------------------------
print("Remaining NaNs:", merged_df.isna().sum().sum())

# ----------------------------
# Save cleaned dataset
# ----------------------------
merged_df.to_csv(output_path, index=False)

print("Saved cleaned dataset to:", output_path)


Merged shape: (11933, 77)
After restricting to urinary respondents: (5208, 77)
After dropping sparse predictors: (5208, 42)
Remaining NaNs: 0
Saved cleaned dataset to: ../data/final/nhanes_2021_2023.csv


In [3]:


import pandas as pd

# ----------------------------
# Load dataset
# ----------------------------
df = pd.read_csv("../data/final/nhanes_2021_2023.csv")

# ----------------------------
# Keep only required columns
# ----------------------------
cols = [
    "DIQ010",
    "MCQ010",
    "KIQ022",
    "KIQ042",
    "KIQ044",
    "KIQ005",
    "KIQ481",
    "RIAGENDR",
    "RIDAGEYR"
]
df = df[cols].copy()

# ----------------------------
# Drop rows with ANY NaNs
# ----------------------------
df = df.dropna()

# ----------------------------
# Convert column names to CAPS
# ----------------------------
df.columns = [c.upper() for c in df.columns]

# ----------------------------
# Symptom BINARISATION
# ----------------------------

# Frequency
df["KIQ005_BIN"] = df["KIQ005"].isin([3,4,5]).astype(int)

# Urgency
df["KIQ044_BIN"] = (df["KIQ044"] == 1).astype(int)

# Nocturia
df["KIQ481_BIN"] = df["KIQ481"].isin([2,3,4,5]).astype(int)

# ----------------------------
# YES/NO BINARISATION (1 = Yes → 1, 2 = No → 0)
# ----------------------------
binary_map = {1: 1, 2: 0}

df["DIQ010_BIN"] = df["DIQ010"].map(binary_map)
df["MCQ010_BIN"] = df["MCQ010"].map(binary_map)
df["KIQ022_BIN"] = df["KIQ022"].map(binary_map)
df["KIQ042_BIN"] = df["KIQ042"].map(binary_map)
df["RIAGENDR_BIN"] = df["RIAGENDR"].map(binary_map)

# ----------------------------
# OAB LABEL
# ----------------------------
df["OAB"] = (
    (df["KIQ044_BIN"] == 1) &
    ((df["KIQ005_BIN"] == 1) | (df["KIQ481_BIN"] == 1))
).astype(int)
# ----------------------------
# Drop original non-binary columns
# ----------------------------
df = df.drop(columns=[
    "DIQ010",
    "MCQ010",
    "KIQ022",
    "KIQ042",
    "KIQ044",
    "KIQ005",
    "KIQ481",
    "RIAGENDR"
])

df = df.dropna()

# ----------------------------
# Save final dataset
# ----------------------------
output_path = "../data/final/full_cleaned_labelled.csv"
df.to_csv(output_path, index=False)

# ----------------------------
# Dataset Stats
# ----------------------------
print("\nFinal Dataset Shape:", df.shape)

counts = df["OAB"].value_counts().sort_index()
print("\nOAB Counts:")
print(counts)

print("\nOAB Prevalence (%):")
print((counts / len(df)) * 100)

print(f"\nSaved to: {output_path}")



Final Dataset Shape: (5034, 10)

OAB Counts:
OAB
0    3554
1    1480
Name: count, dtype: int64

OAB Prevalence (%):
OAB
0    70.599921
1    29.400079
Name: count, dtype: float64

Saved to: ../data/final/full_cleaned_labelled.csv
